# Seminar 06: VQ-VAE-2

**Date**: 2025/02/18

This notebook demonstrates a complete implementation of a VQ-VAE-2 model.




| ![](imgs/VQ-VAE-2_Architecture.png) | 
|:--:| 
| *[Source](https://arxiv.org/abs/1906.00446)* |

## Environment Setup

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms, datasets, utils
from tqdm.auto import tqdm

# Set device
cuda = torch.cuda.is_available()
device = torch.device("cuda" if cuda else ("mps" if torch.backends.mps.is_available() else "cpu"))
torch.backends.cudnn.benchmark = cuda

print(f"Using device: {device}")

## Model Components

We define a custom `Quantize` module (without distributed synchronization), a residual block, the encoder/decoder networks, and finally the full VQ-VAE model.

In [ ]:
# Quantize module (vector quantization)
class Quantize(nn.Module):
    def __init__(self, dim, n_embed, decay=0.99, eps=1e-5):
        super().__init__()
        self.dim = dim
        self.n_embed = n_embed
        self.decay = decay
        self.eps = eps

        embed = torch.randn(dim, n_embed)
        self.register_buffer("embed", embed)
        self.register_buffer("cluster_size", torch.zeros(n_embed))
        self.register_buffer("embed_avg", embed.clone())

    def forward(self, input):
        # input: (B, H, W, dim)
        flatten = input.reshape(-1, self.dim)
        # Compute L2 distance between input vectors and embedding vectors.
        dist = (
            flatten.pow(2).sum(1, keepdim=True)
            - 2 * flatten @ self.embed
            + self.embed.pow(2).sum(0, keepdim=True)
        )
        # Find the nearest embedding for each input
        _, embed_ind = (-dist).max(1)
        embed_onehot = F.one_hot(embed_ind, self.n_embed).type(flatten.dtype)
        embed_ind = embed_ind.view(*input.shape[:-1])
        quantize = self.embed_code(embed_ind)

        if self.training:
            # Update the embedding vectors using exponential moving average
            embed_onehot_sum = embed_onehot.sum(0)
            embed_sum = flatten.transpose(0, 1) @ embed_onehot

            self.cluster_size.mul_(self.decay).add_(embed_onehot_sum, alpha=1 - self.decay)
            self.embed_avg.mul_(self.decay).add_(embed_sum, alpha=1 - self.decay)
            n = self.cluster_size.sum()
            cluster_size = ((self.cluster_size + self.eps) / (n + self.n_embed * self.eps)) * n
            embed_normalized = self.embed_avg / cluster_size.unsqueeze(0)
            self.embed.copy_(embed_normalized)

        diff = (quantize.detach() - input).pow(2).mean()
        # Straight-through estimator
        quantize = input + (quantize - input).detach()
        return quantize, diff, embed_ind

    def embed_code(self, embed_id):
        # Look up embedding vectors (note the transpose to get [*, dim])
        return F.embedding(embed_id, self.embed.t())

In [ ]:
# A simple residual block
class ResBlock(nn.Module):
    def __init__(self, in_channel, channel):
        super().__init__()
        self.block = nn.Sequential(
            nn.ReLU(),
            nn.Conv2d(in_channel, channel, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(channel, in_channel, kernel_size=1),
        )
    def forward(self, x):
        return x + self.block(x)

In [ ]:
# Encoder
class Encoder(nn.Module):
    def __init__(self, in_channel, channel, n_res_block, n_res_channel, stride):
        super().__init__()
        if stride == 4:
            layers = [
                nn.Conv2d(in_channel, channel // 2, kernel_size=4, stride=2, padding=1),
                nn.ReLU(inplace=True),
                nn.Conv2d(channel // 2, channel, kernel_size=4, stride=2, padding=1),
                nn.ReLU(inplace=True),
                nn.Conv2d(channel, channel, kernel_size=3, padding=1),
            ]
        elif stride == 2:
            layers = [
                nn.Conv2d(in_channel, channel // 2, kernel_size=4, stride=2, padding=1),
                nn.ReLU(inplace=True),
                nn.Conv2d(channel // 2, channel, kernel_size=3, padding=1),
            ]
        for _ in range(n_res_block):
            layers.append(ResBlock(channel, n_res_channel))
        layers.append(nn.ReLU(inplace=True))
        self.blocks = nn.Sequential(*layers)
    def forward(self, x):
        return self.blocks(x)

In [ ]:
# Decoder
class Decoder(nn.Module):
    def __init__(self, in_channel, out_channel, channel, n_res_block, n_res_channel, stride):
        super().__init__()
        layers = [nn.Conv2d(in_channel, channel, kernel_size=3, padding=1)]
        for _ in range(n_res_block):
            layers.append(ResBlock(channel, n_res_channel))
        layers.append(nn.ReLU(inplace=True))
        if stride == 4:
            layers.extend([
                nn.ConvTranspose2d(channel, channel // 2, kernel_size=4, stride=2, padding=1),
                nn.ReLU(inplace=True),
                nn.ConvTranspose2d(channel // 2, out_channel, kernel_size=4, stride=2, padding=1)
            ])
        elif stride == 2:
            layers.append(nn.ConvTranspose2d(channel, out_channel, kernel_size=4, stride=2, padding=1))
        self.blocks = nn.Sequential(*layers)
    def forward(self, x):
        return self.blocks(x)

In [ ]:
# The complete VQ-VAE model (a two–stage hierarchical architecture)
class VQVAE(nn.Module):
    def __init__(self,
                 in_channel=3,
                 channel=128,
                 n_res_block=2,
                 n_res_channel=32,
                 embed_dim=64,
                 n_embed=512,
                 decay=0.99):
        super().__init__()
        # Bottom encoder: produces a coarse latent map.
        self.enc_b = Encoder(in_channel, channel, n_res_block, n_res_channel, stride=4)
        # Top encoder: further compresses the bottom encoder output.
        self.enc_t = Encoder(channel, channel, n_res_block, n_res_channel, stride=2)
        # Project top encoding to embedding dimension.
        self.quantize_conv_t = nn.Conv2d(channel, embed_dim, kernel_size=1)
        self.quantize_t = Quantize(embed_dim, n_embed, decay=decay)
        # Top decoder.
        self.dec_t = Decoder(embed_dim, embed_dim, channel, n_res_block, n_res_channel, stride=2)
        # Combine top and bottom features.
        self.quantize_conv_b = nn.Conv2d(embed_dim + channel, embed_dim, kernel_size=1)
        self.quantize_b = Quantize(embed_dim, n_embed, decay=decay)
        # Upsample top quantization to match bottom resolution.
        self.upsample_t = nn.ConvTranspose2d(embed_dim, embed_dim, kernel_size=4, stride=2, padding=1)
        # Final decoder.
        self.dec = Decoder(embed_dim + embed_dim, in_channel, channel, n_res_block, n_res_channel, stride=4)

    def forward(self, x):
        quant_t, quant_b, diff, _, _ = self.encode(x)
        dec = self.decode(quant_t, quant_b)
        return dec, diff

    def encode(self, x):
        # Bottom encoding
        enc_b = self.enc_b(x)   # shape: [B, channel, H/4, W/4]
        # Top encoding from bottom features
        enc_t = self.enc_t(enc_b)  # shape: [B, channel, H/8, W/8]
        # Project and quantize top latent
        quant_t = self.quantize_conv_t(enc_t)  # [B, embed_dim, H/8, W/8]
        # Rearrange to [B, H/8, W/8, embed_dim]
        quant_t = quant_t.permute(0, 2, 3, 1)
        quant_t, diff_t, id_t = self.quantize_t(quant_t)
        quant_t = quant_t.permute(0, 3, 1, 2)
        # Decode top latent to condition bottom
        dec_t = self.dec_t(quant_t)
        # Concatenate decoded top with bottom encoding
        enc_b_cat = torch.cat([dec_t, enc_b], dim=1)
        quant_b = self.quantize_conv_b(enc_b_cat)
        quant_b = quant_b.permute(0, 2, 3, 1)
        quant_b, diff_b, id_b = self.quantize_b(quant_b)
        quant_b = quant_b.permute(0, 3, 1, 2)
        return quant_t, quant_b, diff_t + diff_b, id_t, id_b

    def decode(self, quant_t, quant_b):
        # Upsample the top latent and concatenate with bottom latent
        upsample_t = self.upsample_t(quant_t)
        quant = torch.cat([upsample_t, quant_b], dim=1)
        dec = self.dec(quant)
        return dec

    def decode_code(self, code_t, code_b):
        # Given latent indices, obtain the corresponding embedding vectors and decode.
        quant_t = self.quantize_t.embed_code(code_t).permute(0, 3, 1, 2)
        quant_b = self.quantize_b.embed_code(code_b).permute(0, 3, 1, 2)
        dec = self.decode(quant_t, quant_b)
        return dec

## Training and Inference Functions

Below we define a simple training loop for the VQ-VAE and a helper function for visualizing reconstructions.

In [ ]:
def train_vqvae(model, dataloader, optimizer, device, epochs=10, latent_loss_weight=0.25, sample_interval=500, sample_dir='samples'):
    os.makedirs(sample_dir, exist_ok=True)
    criterion = nn.MSELoss()
    model.train()
    global_step = 0

    for epoch in range(epochs):
        pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{epochs}")
        for images, _ in pbar:
            images = images.to(device)
            optimizer.zero_grad()
            recon, diff = model(images)
            recon_loss = criterion(recon, images)
            loss = recon_loss + latent_loss_weight * diff
            loss.backward()
            optimizer.step()

            pbar.set_postfix({'loss': loss.item(), 'recon': recon_loss.item(), 'latent': diff.item()})
            global_step += 1

            # Save sample reconstructions every sample_interval steps.
            if global_step % sample_interval == 0:
                model.eval()
                with torch.no_grad():
                    recon_sample, _ = model(images)
                    # Concatenate original and reconstruction for comparison
                    comparison = torch.cat([images, recon_sample])
                    utils.save_image(comparison, os.path.join(sample_dir, f'sample_{global_step:06d}.png'),
                                      nrow=images.size(0), normalize=True, range=(-1, 1))
                model.train()

In [ ]:
def infer_vqvae(model, dataloader, device, num_batches=1):
    model.eval()
    reconstructions = []
    with torch.no_grad():
        for i, (images, _) in enumerate(dataloader):
            images = images.to(device)
            recon, _ = model(images)
            reconstructions.append(recon.cpu())
            if i >= num_batches - 1:
                break
    return torch.cat(reconstructions, dim=0)

## Data Preparation

For demonstration we use CIFAR10.

In [ ]:
# Define transforms: here we resize to 64x64 and normalize to roughly [-1,1]
image_size = 32
transform = transforms.Compose([
    transforms.Resize(image_size),
    transforms.CenterCrop(image_size),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

In [ ]:
batch_size = 32
dataset = datasets.CIFAR10(root='../data', download=True, transform=transform)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=4*cuda, pin_memory=cuda)

## Model Setup

Initialize the model, optimizer and (optionally) a learning rate scheduler.

In [ ]:
model = VQVAE(in_channel=3, channel=128, n_res_block=2, n_res_channel=32,
              embed_dim=64, n_embed=512).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
# (Optionally, you could use a PyTorch scheduler such as:
# scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=1000)
# and call scheduler.step() each training iteration.)

## Training

Train the model for a number of epochs. (For demonstration we train only a few epochs; in practice you may need many more.)

_Tip:_ You can change the number of epochs and the frequency of saving sample reconstructions.

In [ ]:
train_vqvae(model, dataloader, optimizer, device, epochs=5, sample_interval=200)

## Inference and Visualization

Let’s see how the model reconstructs some images from the test set.

In [ ]:
# Prepare a dataloader for inference (using the test split of CIFAR10)
test_dataset = datasets.CIFAR10(root='../data', train=False, download=True, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=2)

# Get some reconstructions
recons = infer_vqvae(model, test_loader, device, num_batches=2)

# Save and display the result
utils.save_image(recons, 'reconstructions.png', nrow=8, normalize=True, range=(-1, 1))
print("Saved reconstructions to 'reconstructions.png'.")

## Saving and Loading the Model

You can save the trained model’s state dictionary and later reload it for inference.

```python
torch.save(model.state_dict(), 'vqvae_checkpoint.pt')

# To load:
model = VQVAE(...)  # (initialize with the same parameters)
model.load_state_dict(torch.load('vqvae_checkpoint.pt'))
model.to(device)
model.eval()
```

## Autoregressive Priors: PixelCNN for Discrete Latent Codes

In [ ]:
# Masked Convolution (to enforce autoregressive ordering)
class MaskedConv2d(nn.Conv2d):
    def __init__(self, mask_type, in_channels, out_channels, kernel_size, stride=1, padding=0, bias=True):
        super(MaskedConv2d, self).__init__(in_channels, out_channels, kernel_size, stride, padding, bias=bias)
        self.register_buffer('mask', torch.ones_like(self.weight))
        _, _, kH, kW = self.weight.size()
        self.mask.fill_(1)
        center_h = kH // 2
        center_w = kW // 2
        if mask_type == 'A':
            # For first layer: do not see the current pixel
            self.mask[:, :, center_h, center_w:] = 0
            self.mask[:, :, center_h+1:] = 0
        elif mask_type == 'B':
            # For subsequent layers: allow current pixel but not future ones
            self.mask[:, :, center_h, center_w+1:] = 0
            self.mask[:, :, center_h+1:] = 0

    def forward(self, x):
        self.weight.data *= self.mask
        return super(MaskedConv2d, self).forward(x)

In [ ]:
# Unconditional PixelCNN Prior for Top Latent Codes
class PixelCNNPrior(nn.Module):
    def __init__(self, n_embed, hidden_channels=64, n_layers=7, kernel_size=7):
        super(PixelCNNPrior, self).__init__()
        self.n_embed = n_embed
        self.embedding = nn.Embedding(n_embed, hidden_channels)
        padding = kernel_size // 2
        self.layers = nn.ModuleList()
        # First layer: mask type 'A'
        self.layers.append(MaskedConv2d('A', hidden_channels, hidden_channels, kernel_size, padding=padding))
        # Next layers: mask type 'B'
        for _ in range(n_layers - 1):
            self.layers.append(MaskedConv2d('B', hidden_channels, hidden_channels, kernel_size, padding=padding))
        self.output_conv = nn.Conv2d(hidden_channels, n_embed, kernel_size=1)

    def forward(self, x):
        # x: [B, H, W]
        x_emb = self.embedding(x)
        x_emb = x_emb.permute(0, 3, 1, 2).contiguous()
        out = x_emb
        for layer in self.layers:
            out = F.relu(layer(out))
        logits = self.output_conv(out)
        return logits

In [ ]:
# Conditional PixelCNN Prior for Bottom Latent Codes.
# This model conditions on the top latent codes (which must be upsampled to bottom resolution).
class ConditionalPixelCNNPrior(nn.Module):
    def __init__(self, n_embed, hidden_channels=64, n_layers=7, kernel_size=7, cond_embed_dim=16):
        super(ConditionalPixelCNNPrior, self).__init__()
        self.n_embed = n_embed
        self.embedding = nn.Embedding(n_embed, hidden_channels)
        self.cond_embedding = nn.Embedding(n_embed, cond_embed_dim)
        # Combine latent and condition embeddings
        self.conv_in = nn.Conv2d(hidden_channels + cond_embed_dim, hidden_channels, kernel_size=1)
        padding = kernel_size // 2
        self.layers = nn.ModuleList()
        self.layers.append(MaskedConv2d('A', hidden_channels, hidden_channels, kernel_size, padding=padding))
        for _ in range(n_layers - 1):
            self.layers.append(MaskedConv2d('B', hidden_channels, hidden_channels, kernel_size, padding=padding))
        self.output_conv = nn.Conv2d(hidden_channels, n_embed, kernel_size=1)

    def forward(self, x, condition):
        # x: bottom latent codes, shape [B, H, W] (LongTensor)
        # condition: conditioning codes (upsampled top latent), shape [B, H, W] (LongTensor)
        x_emb = self.embedding(x)       # [B, H, W, hidden_channels]
        x_emb = x_emb.permute(0, 3, 1, 2) # [B, hidden_channels, H, W]
        cond_emb = self.cond_embedding(condition)  # [B, H, W, cond_embed_dim]
        cond_emb = cond_emb.permute(0, 3, 1, 2).contiguous()      # [B, cond_embed_dim, H, W]
        x_cat = torch.cat([x_emb, cond_emb], dim=1)   # [B, hidden_channels+cond_embed_dim, H, W]
        out = F.relu(self.conv_in(x_cat))
        for layer in self.layers:
            out = F.relu(layer(out))
        logits = self.output_conv(out)  # [B, n_embed, H, W]
        return logits

## Training the Autoregressive Priors

We first extract discrete latent codes from the trained VQ-VAE. The following dataset classes help build 
datasets of latent codes for the top and bottom levels.

In [ ]:
class LatentCodeDataset(torch.utils.data.Dataset):
    def __init__(self, base_dataset, vqvae_model, level='top'):
        self.base_dataset = base_dataset
        self.vqvae_model = vqvae_model
        self.level = level  # 'top' or 'bottom'
    def __len__(self):
        return len(self.base_dataset)
    def __getitem__(self, idx):
        image, _ = self.base_dataset[idx]
        image = image.unsqueeze(0).to(device)  # add batch dimension
        with torch.no_grad():
            _, _, _, id_top, id_bottom = self.vqvae_model.encode(image)
        if self.level == 'top':
            latent = id_top.squeeze(0)    # shape: [H_top, W_top]
        else:
            latent = id_bottom.squeeze(0) # shape: [H_bottom, W_bottom]
        return latent.cpu()

In [ ]:
# Training function for an unconditional prior (for top latent codes)
def train_prior(prior_model, dataset, num_epochs=10, batch_size=32, lr=1e-3, device='cpu'):
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    optimizer = torch.optim.Adam(prior_model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    prior_model.to(device)
    prior_model.train()
    for epoch in range(num_epochs):
        pbar = tqdm(dataloader, desc=f"Prior Training Epoch {epoch+1}/{num_epochs}")
        for latent_codes in pbar:
            # latent_codes: [B, H, W] (LongTensor)
            latent_codes = latent_codes.to(device)
            logits = prior_model(latent_codes)  # [B, n_embed, H, W]
            loss = criterion(logits, latent_codes)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            pbar.set_postfix({'loss': loss.item()})
    return prior_model

In [ ]:
# For the conditional prior, we need both bottom codes and the corresponding top codes upsampled to the bottom resolution.
class ConditionalLatentCodeDataset(torch.utils.data.Dataset):
    def __init__(self, base_dataset, vqvae_model):
        self.base_dataset = base_dataset
        self.vqvae_model = vqvae_model
    def __len__(self):
        return len(self.base_dataset)
    def __getitem__(self, idx):
        image, _ = self.base_dataset[idx]
        image = image.unsqueeze(0).to(device)
        with torch.no_grad():
            _, _, _, id_top, id_bottom = self.vqvae_model.encode(image)
        id_top = id_top.squeeze(0)      # shape: [H_top, W_top]
        id_bottom = id_bottom.squeeze(0)  # shape: [H_bottom, W_bottom]
        # Upsample top latent to match bottom latent spatial dimensions using nearest neighbor
        id_top = id_top.unsqueeze(0).unsqueeze(0).float()  # [1,1,H_top,W_top]
        id_top_upsampled = F.interpolate(id_top, size=id_bottom.shape, mode='nearest').squeeze(0).squeeze(0).long()
        return id_bottom.cpu(), id_top_upsampled.cpu()

def train_conditional_prior(prior_model, dataset, num_epochs=10, batch_size=32, lr=1e-3, device='cpu'):
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    optimizer = torch.optim.Adam(prior_model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    prior_model.to(device)
    prior_model.train()
    for epoch in range(num_epochs):
        pbar = tqdm(dataloader, desc=f"Conditional Prior Training Epoch {epoch+1}/{num_epochs}")
        for bottom_codes, top_codes in pbar:
            # bottom_codes: [B, H_bottom, W_bottom], top_codes: [B, H_bottom, W_bottom]
            bottom_codes = bottom_codes.to(device)
            top_codes = top_codes.to(device)
            logits = prior_model(bottom_codes, top_codes)  # [B, n_embed, H_bottom, W_bottom]
            loss = criterion(logits, bottom_codes)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            pbar.set_postfix({'loss': loss.item()})
    return prior_model

In [ ]:
# Train the top (unconditional) prior:
top_latent_dataset = LatentCodeDataset(dataset, model, level='top')
top_prior = PixelCNNPrior(n_embed=512, hidden_channels=64, n_layers=7, kernel_size=7)
top_prior = train_prior(top_prior, top_latent_dataset, num_epochs=10, batch_size=32, lr=1e-3, device=device)

# Train the bottom (conditional) prior:
cond_dataset = ConditionalLatentCodeDataset(dataset, model)
bottom_prior = ConditionalPixelCNNPrior(n_embed=512, hidden_channels=64, n_layers=7, kernel_size=7, cond_embed_dim=16)
bottom_prior = train_conditional_prior(bottom_prior, cond_dataset, num_epochs=10, batch_size=32, lr=1e-3, device=device)

## Sampling and Generating Images

To generate images, we:
1. Sample the top latent codes using the unconditional PixelCNN.
2. Upsample these codes to obtain conditioning for the bottom prior.
3. Sample bottom latent codes with the conditional PixelCNN.
4. Decode the pair using the VQ-VAE decoder.

In [ ]:
def sample_from_prior(prior_model, shape, device='cpu'):
    # shape: (B, H, W)
    B, H, W = shape
    samples = torch.zeros((B, H, W), dtype=torch.long, device=device)
    prior_model.eval()
    with torch.no_grad():
        for i in range(H):
            for j in range(W):
                logits = prior_model(samples)  # [B, n_embed, H, W]
                probs = F.softmax(logits[:, :, i, j], dim=-1)  # [B, n_embed]
                samples[:, i, j] = torch.multinomial(probs, num_samples=1).squeeze(-1)
    return samples

def sample_from_conditional_prior(prior_model, shape, condition, device='cpu'):
    # shape: (B, H, W) for bottom latent codes
    B, H, W = shape
    samples = torch.zeros((B, H, W), dtype=torch.long, device=device)
    prior_model.eval()
    with torch.no_grad():
        for i in range(H):
            for j in range(W):
                logits = prior_model(samples, condition)  # [B, n_embed, H, W]
                probs = F.softmax(logits[:, :, i, j], dim=-1)  # [B, n_embed]
                samples[:, i, j] = torch.multinomial(probs, num_samples=1).squeeze(-1)
    return samples

def generate_images(vqvae_model, top_prior, bottom_prior, batch_size=16, device='cpu'):
    # For our CIFAR10 example:
    # Top latent shape is (B, 4, 4) since image_size=32 -> 32/8=4
    top_shape = (batch_size, 4, 4)
    sampled_top = sample_from_prior(top_prior, top_shape, device=device)
    # Upsample top latent to bottom latent resolution (32/4 = 8)
    top_upsampled = F.interpolate(sampled_top.unsqueeze(1).float(), size=(8, 8), mode='nearest').squeeze(1).long()
    bottom_shape = (batch_size, 8, 8)
    sampled_bottom = sample_from_conditional_prior(bottom_prior, bottom_shape, top_upsampled, device=device)
    # Decode the latents to an image
    with torch.no_grad():
        generated = vqvae_model.decode_code(sampled_top, sampled_bottom)
    return generated

In [ ]:
# Generate new images:
samples = generate_images(model, top_prior, bottom_prior, batch_size=16, device=device)
utils.save_image(samples, 'generated_samples.png', nrow=4, normalize=True, range=(-1, 1))
print("Saved generated samples to 'generated_samples.png'.")